# Creating and Updating Columns

*Calculate derived columns and apply a conditional update to the intended table.*


## Calculate a Column from Existing Values

Assigning to `table["new_name"]` adds a column. Arithmetic on numeric
columns works across the rows, so no explicit row loop is needed.
Keep the original scores and add a separate total for interpretation.


In [1]:
import pandas as pd

score_df = pd.DataFrame({
    "name": ["Minji", "Joon", "Hana", "Taeho"],
    "midterm": [82, 75, 91, 84],
    "final": [88, 79, 94, 86],
    "attendance": [95, 78, 100, 88],
})
score_df["total"] = score_df["midterm"] + score_df["final"]
score_df["passed"] = score_df["total"] >= 160

print("Scores with total and initial pass decision:")
print(score_df)


Scores with total and initial pass decision:
    name  midterm  final  attendance  total  passed
0  Minji       82     88          95    170    True
1   Joon       75     79          78    154   False
2   Hana       91     94         100    185    True
3  Taeho       84     86          88    170    True


Totals are 170, 154, 185, and 170. Minji, Hana, and Taeho initially pass the score rule.


## Update Selected Rows with `loc`

The full rule requires both total score at least 160 and attendance
at least 90. `table.loc[mask, "column"] = value` updates that column
in the selected rows of the original table.


In [2]:
low_attendance_mask = score_df["attendance"] < 90
score_df.loc[low_attendance_mask, "passed"] = False

print("Decisions after the attendance rule:")
print(score_df[["name", "total", "attendance", "passed"]])


Decisions after the attendance rule:
    name  total  attendance  passed
0  Minji    170          95    True
1   Joon    154          78   False
2   Hana    185         100    True
3  Taeho    170          88   False


Taeho changes to `False` because attendance is 88. Joon remains `False`; Minji and Hana pass both requirements.


## Make the Intended Update Explicit

Use `.copy()` when preparing a separate report to edit independently.
In pandas 3, modifying a selected object does not update its parent table.
Chained assignment such as `table[mask]["passed"] = False` cannot update
the original; use the single `.loc[...]` assignment shown above.


In [3]:
report_df = score_df[["name", "total", "passed"]].copy()
report_df["total"] = report_df["total"] + 5

print("Report-only adjusted totals:")
print(report_df[["name", "total"]])
print("Original totals:", score_df["total"].tolist())


Report-only adjusted totals:
    name  total
0  Minji    175
1   Joon    159
2   Hana    190
3  Taeho    175
Original totals: [170, 154, 185, 170]


The separate report shows totals 175, 159, 190, and 175, while the original keeps 170, 154, 185, and 170. The example changes only the report total; it does not recalculate its pass decisions.


Create derived columns with assignment and update selected rows through one `.loc[...]` operation. Use a separate table for independent edits.

[Copy-on-Write in pandas 3](https://pandas.pydata.org/docs/user_guide/copy_on_write.html)
